# DecompDiff — Colab sweep 3 (best config across datasets)

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.

Runs the `local_sweep3` experiments: the fixed **winning objective**
(**MSE loss, predict $x_0$, loss weight ON**, architecture NL1 / NF1 / hidden_dim 64)
across the new datasets and a stock batch-size sweep.

| Cell | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Clone your GitHub repo (must contain `DecompDiff/` and `MyCode/` at its root) |
| 3 | Set paths |
| 4 | Install wandb |
| 5 | wandb login |
| 6 | Imports + experiment function |
| 7 | Run all experiments (6 datasets + stock batch 24/64/128) |
| 8 | (optional) copy checkpoints to Drive |

**One-time setup:** push `DecompDiff/` and `MyCode/` to a GitHub repo, then set `REPO_URL`
in the clone cell below.

## 1. Setup

In [ ]:
# -- Check GPU --------------------------------------------------------------
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"Memory  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# -- Clone repo from GitHub -------------------------------------------------
# The repo MUST contain DecompDiff/ and MyCode/ at its root.
import os
REPO_URL = "https://github.com/Ardameliksah/DecompDiff.git"   # <-- change to your repo
REPO_DIR = "/content/DecompDiff-colab"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

In [ ]:
import os, sys
from pathlib import Path
assert Path(REPO_DIR).exists(), f"Folder not found: {REPO_DIR}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
print("has DecompDiff:", Path("DecompDiff").exists(), "| has MyCode:", Path("MyCode").exists())

In [ ]:
# -- Install missing packages ----------------------------------------------
# Colab already has torch, numpy, pandas, scipy, scikit-learn. Only wandb is missing.
!pip install -q wandb

In [ ]:
# -- Weights & Biases login -------------------------------------------------
# Option A: add a Colab Secret named WANDB_API_KEY (key icon, left sidebar).
# Option B: paste your key when prompted.
import os, wandb
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
except Exception:
    pass
if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"])
else:
    wandb.login()   # interactive prompt
print("wandb:", wandb.__version__)

## 2. Imports & experiment function

In [ ]:
# -- Imports ----------------------------------------------------------------
import os, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import wandb

REPO = Path(REPO_DIR)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from DecompDiff.models.decompDiff import DecompDiff
from DecompDiff.models.diffusion  import GaussianDiffusion
from DecompDiff.config.stocks_config import Config
from DecompDiff.data.datasets import make_loaders
from MyCode.eval_metrics import evaluate_samples, vds_score, fdds_score, correlational_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("REPO:", REPO, "| device:", DEVICE)

In [ ]:
def compute_loss(model, diffusion, x_0, loss_type="mse",
                 prediction_type="x0", use_loss_weight=True):
    B = x_0.shape[0]
    t = torch.randint(0, diffusion.num_timesteps, (B,), device=x_0.device)
    x_t, noise = diffusion.q_sample(x_0, t)
    model_out = model(x_t, t)
    target = x_0 if prediction_type == "x0" else noise
    if loss_type == "l1":
        loss = F.l1_loss(model_out, target, reduction="none")
    else:
        loss = F.mse_loss(model_out, target, reduction="none")
    loss = loss.mean(dim=[1, 2])
    if use_loss_weight:
        loss = loss * diffusion.loss_weight[t]
    return loss.mean()


def compute_inline_metrics(model, diffusion, real_loader, device, num_steps=50,
                           prediction_type="x0"):
    model.eval()
    batches = [b.cpu().numpy() for b in real_loader]
    real_CL = np.concatenate(batches, axis=0)   # (N, C, L)
    N = real_CL.shape[0]
    chunk, chunks = 256, []
    with torch.no_grad():
        for start in range(0, N, chunk):
            bs = min(chunk, N - start)
            chunks.append(
                model.sample(diffusion, batch_size=bs, num_steps=num_steps, eta=0.0,
                             prediction_type=prediction_type).cpu()
            )
    fake_CL = torch.cat(chunks, dim=0).numpy()
    real_m = ((real_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    fake_m = ((fake_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    try:
        results = evaluate_samples(real_m, fake_m, device=device, n_iterations=3,
                                   disc_iterations=2000, pred_iterations=5000)
        vds  = vds_score(real_m, fake_m)
        fdds = fdds_score(real_m, fake_m)
        corr = correlational_score(real_m, fake_m)
    except Exception as e:
        print(f"  [metrics] failed: {e}"); model.train(); return None
    model.train()
    return {
        "disc_score":          results["discriminative"]["mean"],
        "disc_score_std":      results["discriminative"]["std"],
        "test_acc":            results["discriminative"]["test_acc"],
        "pred_mae":            results["predictive"]["mean"],
        "pred_mae_std":        results["predictive"]["std"],
        "vds": vds, "fdds": fdds, "correlational_score": corr,
    }

In [ ]:
def run_experiment(dataset, window_length, num_epochs, num_layers=1,
                   num_fusion_layers=1, hidden_dim=64, batch_size=None,
                   eval_every=500, run_name=None):
    """Winning objective (MSE - predict x0 - loss weight ON) on any dataset."""
    LOSS, PRED, WEIGHT = "mse", "x0", True

    cfg = Config()
    cfg.model.sequence_length = window_length
    cfg.training.num_epochs   = num_epochs
    cfg.data.dataset          = dataset
    cfg.model.num_layers        = num_layers
    cfg.model.num_fusion_layers = num_fusion_layers
    cfg.model.hidden_dim        = hidden_dim
    cfg.training.loss_type       = LOSS
    cfg.training.prediction_type = PRED
    cfg.training.use_loss_weight = WEIGHT
    if batch_size is not None:
        cfg.training.batch_size = batch_size
    bs = cfg.training.batch_size

    train_loader, _, ds = make_loaders(
        dataset, batch_size=bs, window=window_length,
        train_ratio=cfg.data.train_split, neg_one_to_one=cfg.data.neg_one_to_one,
        per_window=cfg.data.per_window_norm, num_workers=cfg.data.num_workers,
        pin_memory=False, data_root=cfg.data.data_root,
        sine_num=cfg.data.sine_num, sine_dim=cfg.data.sine_dim, seed=cfg.data.sine_seed,
    )
    cfg.model.input_channels = ds.num_features

    run_name = run_name or (
        f"decompdiff-{dataset}-L{window_length}-H{hidden_dim}"
        f"-NL{num_layers}-NF{num_fusion_layers}-mse-x0-w1-bs{bs}-E{num_epochs}"
    )

    wandb.init(
        project="decompdiff-sweep3", group="mse-x0-w1-datasets", name=run_name,
        tags=[dataset, f"bs{bs}", f"NL{num_layers}", f"NF{num_fusion_layers}",
              "loss-mse", "pred-x0", "w1"],
        config={**cfg.model.__dict__, **cfg.diffusion.__dict__, **cfg.training.__dict__,
                "dataset": dataset, "device": DEVICE, "eval_every": eval_every},
    )
    print(f"[{run_name}] batches={len(train_loader)} channels={cfg.model.input_channels} "
          f"batch_size={bs}")

    model = DecompDiff(
        input_channels=cfg.model.input_channels, sequence_length=window_length,
        hidden_dim=hidden_dim, num_heads=cfg.model.num_heads, num_layers=num_layers,
        num_fusion_layers=num_fusion_layers, mlp_ratio=cfg.model.mlp_ratio,
        dropout=cfg.model.dropout, freq_dim=cfg.model.freq_dim,
    ).to(DEVICE)
    diffusion = GaussianDiffusion(
        num_timesteps=cfg.diffusion.num_timesteps, beta_start=cfg.diffusion.beta_start,
        beta_end=cfg.diffusion.beta_end, noise_schedule=cfg.diffusion.noise_schedule,
        device=DEVICE,
    ).to(DEVICE)

    counts = model.get_parameter_count()
    print(f"[{run_name}] model_dim={model.model_dim} params={counts['total']:,}")
    wandb.config.update({"total_params": counts["total"], "model_dim": model.model_dim},
                        allow_val_change=True)

    total_steps = len(train_loader) * num_epochs
    optimizer = AdamW(model.parameters(), lr=cfg.training.learning_rate,
                      weight_decay=cfg.training.weight_decay, betas=(0.9, 0.999))
    warmup = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                      total_iters=cfg.training.warmup_steps)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - cfg.training.warmup_steps),
                               eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[cfg.training.warmup_steps])

    ckpt_dir = REPO / f"DecompDiff/output/checkpoints/{run_name}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_every = max(100, num_epochs // 5)

    for epoch in range(num_epochs):
        model.train(); epoch_loss = 0.0
        for batch in train_loader:
            x_0 = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = compute_loss(model, diffusion, x_0, LOSS,
                                prediction_type=PRED, use_loss_weight=WEIGHT)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_val)
            optimizer.step(); scheduler.step(); epoch_loss += loss.item()

        train_loss = epoch_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]
        log = {"train_loss": train_loss, "lr": current_lr, "epoch": epoch + 1}

        if (epoch + 1) % save_every == 0 or (epoch + 1) == num_epochs:
            torch.save({"epoch": epoch + 1, "train_loss": train_loss,
                        "model_state_dict": model.state_dict(),
                        "config": {"model": cfg.model.__dict__, "window_length": window_length,
                                   "dataset": dataset, "prediction_type": PRED,
                                   "use_loss_weight": WEIGHT, "loss_type": LOSS}},
                       ckpt_dir / f"checkpoint_ep{epoch+1}.pt")

        if (epoch + 1) % eval_every == 0:
            print(f"  [epoch {epoch+1}] metrics...")
            metrics = compute_inline_metrics(model, diffusion, train_loader, DEVICE,
                                             num_steps=50, prediction_type=PRED)
            if metrics is not None:
                log.update(metrics)
                print(f"  disc={metrics['disc_score']:.4f} pred={metrics['pred_mae']:.4f} "
                      f"vds={metrics['vds']:.4f} fdds={metrics['fdds']:.4f} "
                      f"corr={metrics['correlational_score']:.4f}")
        wandb.log(log)
        print(f"[{run_name}] ep {epoch+1:4d}/{num_epochs} train={train_loss:.5f} lr={current_lr:.2e}")

    print(f"[{run_name}] done. ckpt -> {ckpt_dir}")
    wandb.finish()
    return model, diffusion

In [ ]:
# -- Shared settings (winning config: MSE - predict x0 - loss weight ON) -----
WINDOW     = 32     # benchmarks (WaveletDiff / Diffusion-TS) use 24 -> set 24 for
                    # apples-to-apples with published numbers.
EPOCHS     = 2000
EVAL_EVERY = 500

## 3. Run all experiments (local_sweep3)

In [ ]:
import gc, traceback

# local_sweep3 experiments: winning config across datasets + stock batch-size sweep
EXPERIMENTS = [
    dict(dataset="etth1"),
    dict(dataset="etth2"),
    dict(dataset="exchange"),
    dict(dataset="fmri"),
    dict(dataset="eeg"),
    dict(dataset="sine"),
    dict(dataset="stock", batch_size=24),
    dict(dataset="stock", batch_size=64),
    dict(dataset="stock", batch_size=128),
]

print(f"device={DEVICE}  total={len(EXPERIMENTS)} experiments  "
      f"(WINDOW={WINDOW}, EPOCHS={EPOCHS})\n")
for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*70}\n  [{i+1}/{len(EXPERIMENTS)}] {exp}\n{'='*70}")
    try:
        run_experiment(dataset=exp["dataset"], window_length=WINDOW, num_epochs=EPOCHS,
                       num_layers=1, num_fusion_layers=1, hidden_dim=64,
                       batch_size=exp.get("batch_size"), eval_every=EVAL_EVERY)
    except Exception:
        print(f"!!! FAILED: {exp}"); traceback.print_exc()
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*70 + "\n  ALL EXPERIMENTS COMPLETE\n" + "="*70)

## 4. (optional) Back up checkpoints to Drive

In [ ]:
# Colab runtimes are ephemeral - checkpoints under /content are lost on disconnect.
# W&B already has your metrics; run this to also copy checkpoints to Drive.
from google.colab import drive; drive.mount("/content/drive")
import shutil
from pathlib import Path
dst = "/content/drive/MyDrive/DecompDiff_output"
shutil.copytree(str(Path(REPO_DIR) / "DecompDiff" / "output"), dst, dirs_exist_ok=True)
print("copied ->", dst)